In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from model_utils.plots import plot_results
from lstm import LSTM
from dataset import TimeSeriesDataset
import time
import itertools

In [ ]:
def lstm_experiment(df, target, train_size=500, val_size=108, forecast_window=153, 
                   seq_length=30, epochs=100, batch_size=32, lr=0.001, dropout=0.0, hidden_size=32, num_layers=1, plot=False, patience=100):

    total_train_val = train_size + val_size
    train = df[target][-(total_train_val+forecast_window):-(val_size+forecast_window)].values
    val = df[target][-(val_size+forecast_window):-forecast_window].values
    test = df[target][-forecast_window:].values
    
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()

    input_size = 1
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"Training LSTM on {device_name} with hidden_size={hidden_size}, batch_size={batch_size}, num_layers={num_layers}, dropout={dropout}")
    
    # Create Datasets
    train_dataset = TimeSeriesDataset(train_scaled, seq_length)
    
    use_pin_memory = torch.cuda.is_available()
    
    # Reduced batch size for more updates per epoch (adjust as needed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=use_pin_memory)
    
    val_dataset = TimeSeriesDataset(val_scaled, seq_length)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, pin_memory=use_pin_memory)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Scheduler to reduce LR when validation loss plateaus
    #scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    train_losses = []
    val_losses = []
    
    best_val_loss = float('inf')
    best_model_path = 'best_lstm_model.pth'
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            # Reshape input to (batch_size, seq_len, input_size) if needed by dataset or model
            batch_x = batch_x.view(-1, seq_length, 1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_train_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                batch_x = batch_x.view(-1, seq_length, 1) # Ensure shape
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        val_losses.append(avg_val_loss)
        
        # Save best model and Early Stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        #if epochs_no_improve >= patience:
        #    print(f"Early stopping triggered at epoch {epoch+1}")
        #    break
        
        # Step the scheduler
        #scheduler.step(avg_val_loss)
        
        if plot and (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
    
    # Load the best model
    model.load_state_dict(torch.load(best_model_path))
    print(f"Loaded best model with val loss: {best_val_loss:.6f}")

    model.eval()
    forecast = []
    
    current_seq = val_scaled[-seq_length:].tolist()
    
    with torch.no_grad():
        for step in range(forecast_window):
            # Ensure input shape matches (1, seq_length, 1) for single prediction
            x = np.array(current_seq[-seq_length:]).reshape(-1, seq_length, 1) 
            x = torch.FloatTensor(x).to(device)
            
            pred = model(x).cpu().numpy()[0, 0]
            forecast.append(pred)
            
            current_seq.append(pred) # Append new prediction
    
    forecast = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    
    if plot:
        train_index = df['date'][-(total_train_val+forecast_window):-(val_size+forecast_window)].values
        val_index = df['date'][-(val_size+forecast_window):-forecast_window].values
        test_index = df['date'][-forecast_window:].values

        plot_results(train, val, test, forecast, train_index, val_index, test_index, train_losses, val_losses, target)
    
    return model, scaler, forecast, rmse, mae, test

In [3]:
ITEM_ID = 26008       # Example item_id from sample_head.csv
STORE_ID = 6269        # Example store_id from sample_head.csv
DATA_PATH = '../dataset/data_andre.feather' # Adjust path if needed
TARGET_COL = 'value'
DATE_COL = 'date'

# To converge in fewer epochs (even if each epoch takes longer), we:
# 1. Decrease Batch Size -> More updates per epoch
# 2. Can increase Hidden Size slightly -> More capacity
# 3. Increase learning rate slightly but use scheduler (already added)

EPOCHS = 1000            # Set a bit higher to ensure convergence, though fewer should be needed
BATCH_SIZE = 4         # Smaller batch size for more frequent updates
LEARNING_RATE = 0.001  # Standard LR
HIDDEN_SIZE = 64      # Increased capacity

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


In [4]:
# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../dataset/subset_set.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df_full = pd.read_feather(DATA_PATH)

print(f"Filtering for Item: {ITEM_ID}, Store: {STORE_ID}...")
# Filter for specific item and store
df = df_full[(df_full['item_id'] == ITEM_ID) & (df_full['store_id'] == STORE_ID)].copy()

# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]

print(len(y))


# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 500
val_size = 108
forecast_horizon = 153

lookback_window = 30

df

Loading data from ../dataset/subset_set.feather...
Filtering for Item: 26008, Store: 6269...
761


,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,promo_value_GAS,...,promo_type_CIRC,promo_value_CIRC,promo_type_CIRE,promo_value_CIRE,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id,date
0,26008,104,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,2021-01-23
1,26008,118,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,2021-01-24
2,26008,116,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,2021-01-25
3,26008,51,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,2021-01-26
4,26008,91,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,2021-01-27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
756,26008,134,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,2023-02-18
757,26008,90,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,2023-02-19
758,26008,79,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,2023-02-20
759,26008,60,sprd btr mrgrn,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,2023-02-21


In [5]:
'''
print(f"Training LSTM with train_size={train_size}, val_size={val_size}, forecast_horizon={forecast_horizon}, lookback_window={lookback_window}...")

# Increased model complexity (num_layers=2) to help learn faster per epoch, even if epoch is slower
model, scaler, forecast, rmse, mae, test = lstm_experiment(
    df=df, 
    target=TARGET_COL, 
    train_size=train_size, 
    val_size=val_size,
    forecast_window=forecast_horizon, 
    seq_length=lookback_window,
    hidden_size=HIDDEN_SIZE, 
    num_layers=1,           # Increased number of layers
    dropout=0.0,            # Added dropout for regularization with more layers
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    lr=LEARNING_RATE, 
    plot=True
)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
'''

'\nprint(f"Training LSTM with train_size={train_size}, val_size={val_size}, forecast_horizon={forecast_horizon}, lookback_window={lookback_window}...")\n\n# Increased model complexity (num_layers=2) to help learn faster per epoch, even if epoch is slower\nmodel, scaler, forecast, rmse, mae, test = lstm_experiment(\n    df=df, \n    target=TARGET_COL, \n    train_size=train_size, \n    val_size=val_size,\n    forecast_window=forecast_horizon, \n    seq_length=lookback_window,\n    hidden_size=HIDDEN_SIZE, \n    num_layers=1,           # Increased number of layers\n    dropout=0.0,            # Added dropout for regularization with more layers\n    epochs=EPOCHS, \n    batch_size=BATCH_SIZE, \n    lr=LEARNING_RATE, \n    plot=True\n)\n\nprint(f"RMSE: {rmse:.4f}")\nprint(f"MAE: {mae:.4f}")\n'

# Grid Search Cell

In [6]:
import time
import itertools
import sys
import os
import importlib
sys.path.append(os.path.abspath('..'))

# Reload plot_results to pick up changes in model_utils.utils
import model_utils.plots
importlib.reload(model_utils.plots)
from model_utils.plots import plot_results

    
# Modify lstm_experiment to return timings and handle plotting internally
def lstm_experiment_grid(df, target, item_id, store_id, train_size=500, val_size=100, forecast_window=161, 
                   seq_length=30, epochs=100, batch_size=32, lr=0.001, dropout=0.0, hidden_size=32, num_layers=1, 
                   patience=50, seed=42, save_plot_path=None):

    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    total_train_val = train_size + val_size
    train = df[target][-(total_train_val+forecast_window):-(val_size+forecast_window)].values
    val = df[target][-(val_size+forecast_window):-forecast_window].values
    test = df[target][-forecast_window:].values
    
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()

    input_size = 1
    
    # Create Datasets
    # dataset.py: def __init__(self, target_data, exog_data, seq_length):
    train_dataset = TimeSeriesDataset(train_scaled, None, seq_length)
    use_pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=use_pin_memory)
    
    val_dataset = TimeSeriesDataset(val_scaled, None, seq_length)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, pin_memory=use_pin_memory)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    val_losses = []
    
    best_val_loss = float('inf')
    
    # Create directory for models if it doesn't exist
    model_dir = f'best_models/seed_{seed}'
    os.makedirs(model_dir, exist_ok=True)
    best_model_path = f'{model_dir}/lstm_item{item_id}_store{store_id}.pth'

    epochs_no_improve = 0
    best_epoch = 0

    # Training Time
    start_train_time = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            batch_x = batch_x.view(-1, seq_length, 1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_train_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                batch_x = batch_x.view(-1, seq_length, 1) # Ensure shape
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        val_losses.append(avg_val_loss)
        
        # Save best model and Early Stopping
        if avg_val_loss <= best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
       #     epochs_no_improve = 0
            best_epoch = epoch + 1
       # else:
       #     epochs_no_improve += 1
            
       # if epochs_no_improve >= patience:
       #     break
            
    train_time = time.time() - start_train_time

    # Inference Time
    start_inference_time = time.time()
    
    # Load the best model
    model.load_state_dict(torch.load(best_model_path))
    print (f"Loaded best model in path: {best_model_path} with val loss: {best_val_loss:.6f} at epoch {best_epoch}")
    model.eval()
    forecast = []
    
    current_seq = val_scaled[-seq_length:].tolist()
    
    with torch.no_grad():
        for step in range(forecast_window):
            x = np.array(current_seq[-seq_length:]).reshape(-1, seq_length, 1) 
            x = torch.FloatTensor(x).to(device)
            pred = model(x).cpu().numpy()[0, 0]
            forecast.append(pred)
            current_seq.append(pred)
    
    forecast = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    
    inference_time = time.time() - start_inference_time
    
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    
    if save_plot_path:
        # Use the passed dataframe 'df' to get dates, not the global 'df' or unpassed variable
        train_index = df['date'][-(total_train_val+forecast_window):-(val_size+forecast_window)].values
        val_index = df['date'][-(val_size+forecast_window):-forecast_window].values
        test_index = df['date'][-forecast_window:].values
        
        plot_results(train, val, test, forecast, train_index, val_index, test_index, 
                     train_losses, val_losses, target, 
                     title=f'LSTM Forecast (Seed={seed}, Item={item_id}, Store={store_id})',
                     save_path=save_plot_path)
    
    return rmse, mae, train_time, inference_time, best_epoch

# Grid Search Parameters
seeds = [42, 123, 351, 789, 1471, 2024]
batch_size = 32
hidden_size = 64
dropout = 0.0

# Get all unique products from the subset dataset
products = df_full[['item_id', 'store_id']].drop_duplicates().values

# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)

# Run Grid Search
print(f"Starting Grid Search with {len(seeds)} seeds and {len(products)} products...")

for seed in seeds:
    print(f"\n--- Processing Seed: {seed} ---")
    
    for item_id, store_id in products:
        print(f"Running: Seed={seed}, Item={item_id}, Store={store_id}")
        
        # Filter data for the specific product
        df_product = df_full[(df_full['item_id'] == item_id) & (df_full['store_id'] == store_id)].copy()
        
        # Handle DATE_COL (ensure it is a column and not in the index)
        if DATE_COL in df_product.index.names:
            if DATE_COL in df_product.columns:
                # If it's in both, drop the index version to avoid "cannot insert" error
                df_product = df_product.reset_index(drop=True)
            else:
                # If it's only in the index, move it to a column
                df_product = df_product.reset_index()

        # Fallback: simple reset to ensure RangeIndex 0..N
        df_product = df_product.reset_index(drop=True)

        df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
        df_product = df_product.sort_values(DATE_COL)
        df_product = df_product.reset_index(drop=True) # Final clean reset
        
        # Create directory for plots if it doesn't exist
        plot_dir = f'grid_search_plots/seed_{seed}'
        os.makedirs(plot_dir, exist_ok=True)
        plot_filename = f'{plot_dir}/lstm_item{item_id}_store{store_id}.png'
        
        rmse, mae, train_time, infer_time, best_epoch = lstm_experiment_grid(
            df=df_product, 
            target=TARGET_COL, 
            item_id=item_id,
            store_id=store_id,
            train_size=train_size, 
            val_size=val_size,
            forecast_window=forecast_horizon, 
            seq_length=lookback_window,
            epochs=EPOCHS,  # Use the global EPOCHS setting (e.g., 1000)
            batch_size=batch_size, 
            lr=LEARNING_RATE,
            dropout=dropout,
            hidden_size=hidden_size,
            num_layers=1,
            patience=1000,
            seed=seed,
            save_plot_path=plot_filename
        )
        
        results.append({
            'seed': seed,
            'item_id': item_id,
            'store_id': store_id,
            'batch_size': batch_size,
            'hidden_size': hidden_size,
            'dropout': dropout,
            'rmse': rmse,
            'mae': mae,
            'train_time': train_time,
            'inference_time': infer_time,
            'best_epoch': best_epoch,
            'plot_path': plot_filename
        })

# Convert to DataFrame
    results_df = pd.DataFrame(results)
    results_df.to_csv('grid_search_results.csv', index=False)

Starting Grid Search with 6 seeds and 10 products...

--- Processing Seed: 42 ---
Running: Seed=42, Item=26008, Store=6269
Loaded best model in path: best_models/seed_42/lstm_item26008_store6269.pth with val loss: 0.001790 at epoch 484
Running: Seed=42, Item=921558, Store=6269
Loaded best model in path: best_models/seed_42/lstm_item921558_store6269.pth with val loss: 0.019792 at epoch 134
Running: Seed=42, Item=213626, Store=6269
Loaded best model in path: best_models/seed_42/lstm_item213626_store6269.pth with val loss: 0.003985 at epoch 238
Running: Seed=42, Item=213625, Store=6269
Loaded best model in path: best_models/seed_42/lstm_item213625_store6269.pth with val loss: 0.009917 at epoch 90
Running: Seed=42, Item=213624, Store=6269
Loaded best model in path: best_models/seed_42/lstm_item213624_store6269.pth with val loss: 0.011089 at epoch 390
Running: Seed=42, Item=213628, Store=6269
Loaded best model in path: best_models/seed_42/lstm_item213628_store6269.pth with val loss: 0.00701